# Treatment effect by brain region and time point

This notebook makes every figure for the **condition (mtDSB vs control) × region × age (P21 vs P60)**
comparison from the tidy tables in `results/region_time/`. It never touches the h5ad, so it runs on a
laptop. The tables are produced once, where the data is mounted, with:

```bash
conda activate karospace311   # scanpy + pydeseq2 >= 0.5
python scripts/run_region_time_de.py \
    --h5ad /Volumes/processing2/oligo-mtDSB/data/mtDNA_DSB_5k_clustered_annotation_with_rbd_2_cytetype_brain_novae3.h5ad \
    --outdir results/region_time --layer counts --n-cpus 8
```

The statistics live in `utils/region_time_de.py`; the plots in `utils/region_time_plots.py`.
See the README section *Region × time-point comparison* for the design.

| table | question |
|---|---|
| `de_condition_by_region_age.csv` | mtDSB vs control **within** each (cell type, region, age) stratum; pyDESeq2 Wald, sections as replicates |
| `de_age_contrast_by_region.csv` | is the treatment effect different at P60 vs P21? z-test on the difference of the two stratum log2FCs |
| `de_interaction_by_region.csv` | same question, joint model `~ age + condition + age:condition` per (cell type, region); `term == age_x_condition` |
| `de_region_vs_rest.csv` | is the treatment effect in a region different from the rest of the tissue? |
| `summary_by_region_age.csv` | hit counts and activation index per stratum |

To preview the notebook on the synthetic test data, run `python tests/test_region_time_de.py`
and set `RESULTS_DIR = REPO / "tests" / "_tmp_region_time"` below.


In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO))
from utils import region_time_de as rt
from utils import region_time_plots as rp

RESULTS_DIR = REPO / "results" / "region_time"
# RESULTS_DIR = REPO / "tests" / "_tmp_region_time"   # synthetic preview
FIG_DIR = RESULTS_DIR / "figures"
KEYS = rt.Keys()                      # cell_class / RBD_compartment_simplified / age 21,60 / mtDSB vs control
PADJ, LFC, MIN_BASEMEAN = 0.05, 0.5, 20.0

# cell types to show in detail (only those present in the tables are used)
FOCUS = ["Oligodendrocytes", "Oligodendrocyte Precursor Cells", "Astrocytes", "Microglia", "Neural Stem Cells", "OL"]
# genes highlighted throughout (ISR / mitokine / antigen presentation / stress panel from the README)
PANEL = ["Atf5", "Atf4", "Trib3", "Cdkn1a", "Ddit4", "Gdf15", "Adm", "Igfbp3", "Serpina3n", "Cst7", "C4b", "Cd40",
         "Cd74", "B2m", "H2-D1", "H2-K1", "Gpnmb", "Mt1", "Mt2", "Gfap", "Hcrt", "Pmch", "Sqstm1", "Hspa5"]

tables = rt.load_results(RESULTS_DIR)
if not tables:
    raise SystemExit(f"No tables in {RESULTS_DIR}. Run scripts/run_region_time_de.py first (see the cell above).")
print("tables:", {k: v.shape for k, v in tables.items()})
info_path = RESULTS_DIR / "run_info.json"
if info_path.exists():
    print(json.dumps(json.load(open(info_path)), indent=1)[:1500])

se, summ = tables["simple_effects"], tables["summary"]
cell_types = [c for c in FOCUS if c in set(se["cell_type"])] or sorted(se["cell_type"].unique())
print("cell types in focus:", cell_types)


## 0. Which strata could be tested?

A stratum is skipped when either condition has fewer than two sections after the per-section
minimum-cell filter. Skipped strata are the honest boundary of the analysis and should be reported.

In [ ]:
log = tables["simple_effects_log"]
display(log.pivot_table(index=["cell_type", "region"], columns="age", values="status", aggfunc="first"))
print(f"{(log.status == 'ok').sum()} / {len(log)} strata tested")
if "interaction_log" in tables:
    il = tables["interaction_log"]
    print(f"interaction models: {(il.status == 'ok').sum()} / {len(il)} (cell type, region) pairs")


## 1. Overview: how many genes respond, where and when

Counts of genes with padj < 0.05, |log2FC| ≥ 0.5 and baseMean ≥ 20 in each stratum. `All regions` is the
whole-section pseudobulk per cell type (the region-independent effect).

In [ ]:
for value in ["n_sig", "n_up", "n_down", "activation_index"]:
    fig = rp.plot_hits_overview(summ, value=value, keys=KEYS)
    rp.save(fig, FIG_DIR / f"overview_{value}")
    plt.show()


In [ ]:
# table version of the same figure, for the manuscript
wide = summ.pivot_table(index=["cell_type", "region"], columns="age", values=["n_up", "n_down"], aggfunc="first")
display(wide.loc[cell_types])


## 2. P21 vs P60 treatment effect per region

Each point is a gene; x and y are the mtDSB vs control log2FC at the two ages in that region. Colour says
where the gene is significant; a ring marks a significant P60 − P21 difference (z-test on the delta).

In [ ]:
ac = tables["age_contrast"]
for ct in cell_types:
    fig = rp.plot_age_scatter_grid(ac, ct, keys=KEYS, padj_thr=PADJ, forced_genes=PANEL, label_top=8)
    fig.suptitle(ct, x=0.01, ha="left", fontsize=12)
    rp.save(fig, FIG_DIR / f"age_scatter_{ct.replace(' ', '_')}")
    plt.show()


## 3. Which genes, in which region, at which age

Top significant genes per stratum (union), blank where not significant; and the fixed stress / mitokine
panel so that the same genes can be compared across cell types.

In [ ]:
for ct in cell_types:
    fig = rp.plot_gene_heatmap(se, ct, top_n=8, mask_padj=PADJ, min_base_mean=MIN_BASEMEAN, keys=KEYS)
    rp.save(fig, FIG_DIR / f"heatmap_top_{ct.replace(' ', '_')}")
    plt.show()
    panel_present = [g for g in PANEL if g in set(se["gene"])]
    fig = rp.plot_gene_heatmap(se, ct, genes=panel_present, mask_padj=None, keys=KEYS, vmax=3)
    fig.axes[0].set_title(f"{ct}: stress / mitokine panel (all values shown, not masked)", fontsize=9, loc="left")
    rp.save(fig, FIG_DIR / f"heatmap_panel_{ct.replace(' ', '_')}")
    plt.show()


## 4. Formal test: does the treatment effect change between P21 and P60?

Joint pyDESeq2 model per (cell type, region): `~ age + condition + age:condition`. The `age_x_condition`
term is the difference in treatment effect between the two ages. This is the test to quote when saying that
a response is "later" or "progressive".

In [ ]:
it = tables.get("interaction", pd.DataFrame())
if it.empty:
    print("no interaction table (run without --no-interaction)")
else:
    ix = it[it.term == "age_x_condition"]
    hits = ix[(ix.padj < PADJ) & (ix.log2FC.abs() >= LFC) & (ix.baseMean >= MIN_BASEMEAN)]
    counts = hits.groupby(["cell_type", "region"]).size().rename("n_interaction_hits")
    display(counts.to_frame().sort_values("n_interaction_hits", ascending=False).head(30))
    hits.sort_values("padj").to_csv(RESULTS_DIR / "interaction_hits.csv", index=False)
    for ct in cell_types:
        regs = sorted(hits.loc[hits.cell_type == ct, "region"].unique()) or [rt.TISSUE_WIDE]
        for reg in regs[:6]:
            fig = rp.plot_interaction_lollipop(it, ct, reg, keys=KEYS, top_n=20, padj_thr=PADJ)
            rp.save(fig, FIG_DIR / f"interaction_{ct.replace(' ', '_')}_{reg.replace(' ', '_').replace('/', '-')}")
            plt.show()


## 5. Region-specific responses

For each (cell type, region, age): treatment log2FC in the region vs in the rest of the tissue. Genes off the
diagonal with a significant difference respond in that region more (or less) than elsewhere.

In [ ]:
rr = tables.get("region_vs_rest", pd.DataFrame())
if rr.empty:
    print("no region-vs-rest table (run without --no-region-vs-rest)")
else:
    spec = rr[(rr.padj < PADJ) & (rr.delta_log2FC.abs() >= LFC)]
    display(spec.groupby(["cell_type", "age", "region"]).size().rename("n_region_specific").to_frame().query("n_region_specific > 0"))
    spec.sort_values("padj").to_csv(RESULTS_DIR / "region_specific_hits.csv", index=False)
    for ct in cell_types:
        for age in KEYS.ages:
            fig = rp.plot_region_specificity(rr, ct, age, padj_thr=PADJ, label_top=12)
            rp.save(fig, FIG_DIR / f"region_specific_{ct.replace(' ', '_')}_P{age}")
            plt.show()


## 6. Volcanoes for selected strata

In [ ]:
SELECTED = [(ct, reg, age) for ct in cell_types[:2] for reg in ["Thalamus", "Hypothalamus", "Fiber tracts", rt.TISSUE_WIDE] for age in KEYS.ages]
for ct, reg, age in SELECTED:
    one = se[(se.cell_type == ct) & (se.region == reg) & (se.age == age)]
    if one.empty:
        continue
    fig = rp.plot_volcano(one, f"{ct} - {reg} - P{age}", PADJ, LFC, MIN_BASEMEAN, forced_genes=PANEL, label_top=10)
    rp.save(fig, FIG_DIR / f"volcano_{ct.replace(' ', '_')}_{reg.replace(' ', '_')}_P{age}")
    plt.show()


## 7. Markdown summary for `results/paper/`

In [ ]:
lines = ["| cell type | region | P21 up / down | P60 up / down | top up P21 | top up P60 |", "|---|---|---|---|---|---|"]
for (ct, reg), g in summ[summ.cell_type.isin(cell_types)].groupby(["cell_type", "region"], sort=True):
    g = g.set_index("age")
    def cell(a, col):
        return g.loc[a, col] if a in g.index else "–"
    lines.append(f"| {ct} | {reg} | {cell(KEYS.early, 'n_up')} / {cell(KEYS.early, 'n_down')} | "
                 f"{cell(KEYS.late, 'n_up')} / {cell(KEYS.late, 'n_down')} | {cell(KEYS.early, 'top_up')} | {cell(KEYS.late, 'top_up')} |")
out = RESULTS_DIR / "summary_table.md"
out.write_text("\n".join(lines))
print(f"wrote {out}")
print("\n".join(lines[:12]))
